# 02 — Silver Transform: Fleet & Route Performance Analytics
Reads Bronze Delta tables, applies business rules, and writes enriched Silver tables.

| Silver Table | Derivations |
|---|---|
| silver_vehicles | age_years, is_active flag |
| silver_routes | no enrichment needed |
| silver_deliveries | delay_band, load_utilisation_pct, timestamps cast |
| silver_fuel_logs | fuel_efficiency_km_per_litre (via vehicle join) |

In [ ]:
LAKEHOUSE_NAME = "transportation_lakehouse"
BRONZE = "bronze"
SILVER = "silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE_NAME}.{SILVER}")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── silver_vehicles ──────────────────────────────────────────────────────────
df_v = spark.table(f"{LAKEHOUSE_NAME}.{BRONZE}.bronze_vehicles")

df_sv = (
    df_v
    .withColumn("age_years", F.year(F.current_date()) - F.col("year_registered"))
    .withColumn("is_active", (F.col("status") == "Active").cast("int"))
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_sv.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{SILVER}.silver_vehicles")
print(f"silver_vehicles: {df_sv.count():,}")

In [ ]:
# ── silver_routes ────────────────────────────────────────────────────────────
df_r = spark.table(f"{LAKEHOUSE_NAME}.{BRONZE}.bronze_routes").drop("_ingested_at") \
    .withColumn("_updated_at", F.current_timestamp())

df_r.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{SILVER}.silver_routes")
print(f"silver_routes: {df_r.count():,}")

In [ ]:
# ── silver_deliveries ────────────────────────────────────────────────────────
df_d = spark.table(f"{LAKEHOUSE_NAME}.{BRONZE}.bronze_deliveries")
df_rt = spark.table(f"{LAKEHOUSE_NAME}.{BRONZE}.bronze_routes").select("route_id", "capacity_tonnes") \
    if False else spark.table(f"{LAKEHOUSE_NAME}.{BRONZE}.bronze_vehicles").select("vehicle_id", "capacity_tonnes")

df_sd = (
    df_d
    .withColumn("planned_departure", F.to_timestamp("planned_departure"))
    .withColumn("actual_arrival",    F.to_timestamp("actual_arrival"))
    .join(df_rt, on="vehicle_id", how="left")
    .withColumn(
        "delay_band",
        F.when(F.col("delay_hrs") == 0, "On Time")
         .when(F.col("delay_hrs") <= 1, "Minor (<1 h)")
         .when(F.col("delay_hrs") <= 4, "Moderate (1–4 h)")
         .otherwise("Severe (>4 h)")
    )
    .withColumn(
        "load_utilisation_pct",
        F.round(F.col("load_tonnes") / F.col("capacity_tonnes") * 100, 1)
    )
    .drop("_ingested_at")
    .withColumn("_updated_at", F.current_timestamp())
)

df_sd.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{SILVER}.silver_deliveries")
print(f"silver_deliveries: {df_sd.count():,}")

In [ ]:
# ── silver_fuel_logs ─────────────────────────────────────────────────────────
# Approximate km_per_litre by dividing avg distance per delivery by litres
# We compute a proxy: odometer_km doesn't give trip distance, so we use
# total_cost_gbp / litres_filled == cost_per_litre as a data-quality check
df_fl = spark.table(f"{LAKEHOUSE_NAME}.{BRONZE}.bronze_fuel_logs")

# Avg km per delivery per vehicle as proxy
df_del_veh = (
    spark.table(f"{LAKEHOUSE_NAME}.{BRONZE}.bronze_deliveries")
    .groupBy("vehicle_id")
    .agg(F.avg("distance_km").alias("avg_delivery_km"))
)

df_sfl = (
    df_fl
    .join(df_del_veh, on="vehicle_id", how="left")
    .withColumn(
        "fuel_efficiency_km_per_litre",
        F.round(F.col("avg_delivery_km") / F.col("litres_filled"), 3)
    )
    .withColumn("cost_check", F.round(F.col("total_cost_gbp") / F.col("litres_filled"), 3))
    .drop("_ingested_at", "avg_delivery_km")
    .withColumn("_updated_at", F.current_timestamp())
)

df_sfl.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{SILVER}.silver_fuel_logs")
print(f"silver_fuel_logs: {df_sfl.count():,}")

In [ ]:
print("\n=== Silver Transform Complete ===")
for tbl in ["silver_vehicles", "silver_routes", "silver_deliveries", "silver_fuel_logs"]:
    cnt = spark.table(f"{LAKEHOUSE_NAME}.{SILVER}.{tbl}").count()
    print(f"  {tbl}: {cnt:,}")